# 07 Character-level CNN, name and ending apart (Model 2c)

The revised CNN of `06_charcnn2.ipynb` (Model 2b) with two changes at the end of the network. Both aim at the errors the trained Model 2b still makes (family-first setup, on the validation set): about 2,490 missed DGA domains (false negatives, FN) and 1,850 false alarms (false positives, FP: 1.85% of the legitimate domains). Everything else is the same as in 06: the data, the two setups, the loss, the training settings and the metrics, so a difference in the results comes from these two changes.

**Where Model 2b fails.** Most of its errors are in two groups of names, where DGA and legitimate names look alike:

- **Short names.** DGA: qsnatch (`bmn.xyz`), virut (`qbazys.com`), conficker (`bqpgyq.net`), nymaim, proslikefan: about 61% of the missed DGA domains (qsnatch alone 29%). Legitimate: 24% of the legitimate names have 6 characters or fewer (`bisp.net`, `pej86.com`, `fdr.org.br`), and in the latent space of Model 2b the legitimate domains that sit among DGA domains are next to virut, matsnu and qsnatch: the false alarms are the same kind of name.
- **Names made of real words.** matsnu (`buildingsnerve.com`), nymaim2, gozi, bigviktor: about 14% of the missed DGA domains. Their characters are those of legitimate names; only the meaning differs. This notebook does not address them: they need word-level features.

**Why.** For each filter, Model 2b keeps its maximum and its mean over the whole domain, and the length of the whole domain: it does not know where the name stops and the ending (the public suffix: `.com`, `.co.zw`, ...) starts. In a short name the ending is a large part of the domain: in `bmn.xyz` the mean of every filter is half `.xyz`, and the length is 7 instead of 3. The name is at most half of the domain in 54% of the qsnatch domains, against 10% of the legitimate ones. Model 2 (03) kept every position, so it knew where the name stopped, and qsnatch is the family Model 2b got worse on (recall 0.832 -> 0.754). Also, the dropout before the dense layer of Model 2b drops its one length number too: in training the length is 0 half of the time, so the model learns not to rely on it, though it is the best clue for short names.

| Change | Model 2b (06) | Model 2c (this notebook) |
|---|---|---|
| 1. Pooling | for each filter, its maximum and mean over the whole domain, plus log(length): 257 numbers | for each filter, its maximum and mean over the name, and over the ending, plus the length of both parts: 514 numbers |
| 2. Dropout | 0.5 on the 257 numbers, the length included | `POOL_DROPOUT` (0.2) on the pooled numbers only: the lengths always reach the dense layer. After the dense layer: 0.5, as in 06 |

The missed short DGA names and the false alarms on short legitimate names are in the same zone, so if the changes work, **both** go down there. Moving the 0.5 threshold could not do this: it only trades one for the other.

It is run in the two setups of 06: **binary** (target DGA/legitimate) and **family-first** (target `legit` or a DGA family, collapsed with P(DGA) = 1 - P(legit)). The input is the full domain name, TLD included, with the characters of the ending marked; both setups use the sample weights. Metrics are computed on the DGA/legitimate decision, on the validation set; the test set is kept for the final results.

After loading and encoding the data, the notebook follows four steps:

1. **Define a predictor**: Model 2c, which gives each domain a probability for every class.
2. **Define a loss**: weighted cross-entropy plus L1 and L2 penalties on the weights.
3. **Optimize the loss**: Adam on mini-batches, keeping the best epoch on validation.
4. **Test the model**: metrics on the DGA/legitimate decision, compared with Model 2 and Model 2b; the missed DGA domains and false alarms of every model per length of the name; and the latent spaces of Model 2b and Model 2c.

The notebook defines each step and runs it; the code it uses is in `src/models/`:

- `charcnn3.py`: the architecture of Model 2c and its input (`encode_split`); `charcnn2.py` (Model 2b) and `charcnn.py` (Model 2), for the comparison.
- `train.py`: the training loop (`fit`), which saves every epoch and continues an interrupted training.
- `tracking.py`: MLflow tracking of the runs, shared by every notebook.
- `plots.py`: the loss plot, redrawn after every epoch, and the latent-space plots.
- `latent.py`: the latent space of a trained CNN, and the conflicts in it.
- `metrics.py`: the metrics, shared with the baselines, and the errors per group.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

## Settings

Training is the same as in 06: Adam, learning rate 1e-4, batch size 250, L1 1e-5 and L2 1e-4, 40 epochs, keeping the best epoch on validation by macro-F1, and the same size: `WIDTH` filters in every convolution, `HIDDEN` units in the dense layer. The new setting is `POOL_DROPOUT`, the dropout rate on the pooled numbers (Model 2b: 0.5, on the length too); `DROPOUT` is now only the rate after the dense layer. On a CPU one epoch over the full training set takes a long time, so use a GPU (e.g. Colab) for `SAMPLE = None`.

Each setup is saved after every epoch in its folder of `RUN_DIRS`, and tracked as one run of the MLflow experiment `MLFLOW_EXPERIMENT`: `charcnn`, the same as 03 and 06, so the three CNNs are side by side in MLflow. `CNN2_DIRS` and `PAPER_DIRS` are the folders of the trained Model 2b (06) and Model 2 (03), used for the comparison at the end; a model whose folder has no `best.pt` is left out of it. The folders of Model 2b are not committed: copy them into `checkpoints/` from the machine that trained them.

In [ ]:
SAMPLE = None   # None = full training set
EPOCHS = 40
LR = 1e-4          # Adam learning rate
BATCH_SIZE = 250
L1, L2 = 1e-5, 1e-4   # penalties on the convolution and dense weights
WIDTH = 128        # filters in every convolution
HIDDEN = 128       # units in the dense layer
DROPOUT = 0.5      # dropout rate after the dense layer (as in 06)
POOL_DROPOUT = 0.2 # dropout rate on the pooled numbers (not on the lengths)
SEED = 0
EVAL_SET = "val"   # "test" only for the final results
WEIGHTED = True    # use the sample weights in the loss
CHECKPOINTS = ROOT / "checkpoints"   # trained models are saved here (not committed)
RUN = 1            # run number in the folder names below; a new number starts a new run
RUN_DIRS = {"binary": CHECKPOINTS / f"checkpoints_binary_cnn3_{RUN}",
            "family": CHECKPOINTS / f"checkpoints_multifamily_cnn3_{RUN}"}
CNN2_DIRS = {"binary": CHECKPOINTS / "checkpoints_binary_cnn2_1",       # Model 2b (06)
             "family": CHECKPOINTS / "checkpoints_multifamily_cnn2_1"}
CNN2 = {"width": 128, "hidden": 128, "dropout": 0.5}   # the settings of Model 2b in 06
PAPER_DIRS = {"binary": CHECKPOINTS / "checkpoints_binary_cnn_1",       # Model 2 (03)
              "family": CHECKPOINTS / "checkpoints_multifamily_cnn_1"}
MLFLOW_EXPERIMENT = "charcnn"   # stored in mlruns/ (not committed); the same as 03 and 06

In [ ]:
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display

from src.data.dataset import load_splits, sample_weights
from src.models import latent, tracking
from src.models.charcnn import CharCNN       # Model 2, for the comparison
from src.models.charcnn2 import CharCNN2     # Model 2b, for the comparison
from src.models.charcnn3 import ENDING, CharCNN3, encode_split, name_length
from src.models.encoding import class_codes, encode
from src.models.metrics import binary_metrics, errors_by_group, recall_per_family
from src.models.plots import plot_conflicts, plot_latent, plot_losses
from src.models.train import (Split, fit, load_history, p_dga, penalty, pick_device,
                              predict_proba, set_seed)

device = pick_device()
print("device:", device, torch.cuda.get_device_name() if device.type == "cuda" else "")

train_full, val, test = load_splits()
train = train_full.sample(n=SAMPLE, random_state=SEED) if SAMPLE else train_full
evaluate = test if EVAL_SET == "test" else val
y_val = (val["label"] == "dga").astype(int).to_numpy()
y_eval = (evaluate["label"] == "dga").astype(int).to_numpy()
print(f"train: {len(train):,} rows (of {len(train_full):,}), "
      f"{EVAL_SET}: {len(evaluate):,} rows")

## Input encoding

The input is the full domain name, TLD included, as in 06: each character becomes its ASCII code, padded with 0 at the end to 93 characters. One thing is added: the characters of the **ending** get 128 added to their code (`encode_split` in `src/models/charcnn3.py`), so the model knows where the name stops. ASCII codes are below 128, so the character itself is kept.

The ending is the public suffix and the dot before it, as `src/features/split.py` splits the domain for the hand-crafted features of 04 (tldextract, with the suffixes of the alternative DNS systems): `bmn.xyz` -> name `bmn`, ending `.xyz`; `fdr.org.br` -> `fdr` and `.org.br`; `0756293c.duckdns.org` -> `0756293c` and `.duckdns.org`. Every domain of the dataset has an ending, and splitting the 2 million domains takes about a minute.

The model reads the same character with the same embedding in the name and in the ending, so the convolutions read the domain as in 06; the mark only tells the pooling which part a character is in. Model 2 and Model 2b, compared at the end, get their own input (`encode`, no mark).

In [ ]:
x_train, x_val = (encode_split(part["domain"]) for part in (train, val))
x_eval = x_val if EVAL_SET == "val" else encode_split(evaluate["domain"])
classes = {"label": sorted(train_full["label"].unique()),
           "family": sorted(train_full["family"].unique())}
print(f"input: {x_train.shape}, {len(classes['family'])} family classes")
print("bmn.xyz ->", encode_split(["bmn.xyz"])[0][:9], "...   (the ending: 46+128, 120+128, ...)")


def parts(codes):
    """The name and the ending of one encoded domain, as the model sees them."""
    chars = codes[codes != 0]
    return bytes(chars[chars < ENDING]).decode(), bytes(chars[chars >= ENDING] - ENDING).decode()


examples = pd.concat([val[val["family"] == family].head(2)
                      for family in ("qsnatch", "virut", "orchard_subdomain", "legit")])
pd.DataFrame([(domain, family, *parts(codes)) for domain, family, codes
              in zip(examples["domain"], examples["family"], encode_split(examples["domain"]))],
             columns=["domain", "family", "name", "ending"])

## 1. Define a predictor

The predictor is Model 2c (`src/models/charcnn3.py`):

```text
embedding 128 -> 60, index 0 = padding (kept at zero)
conv 128 x 7, conv 128 x 6, conv 128 x 5, conv 128 x 5, conv 128 x 4, conv 128 x 3
    each followed by batch normalization (over the real characters) and a ReLU
max and mean of each filter over the name (2 x 128) and over the ending (2 x 128)
dropout 0.2 on these 512 numbers, then log(1 + length) of the name and of the ending -> 514
dense 128 + ReLU, dropout 0.5, output layer
```

Up to the convolutions it is Model 2b: the convolutions read the whole domain, across the dot, so a filter still sees where the name meets the ending. Only the end differs. What each filter found in the name and what it found in the ending are kept apart, with the length of each part, so a 3-character random name is no longer averaged with its `.xyz`, and the lengths are never dropped out.

Like Model 2b, it maps an encoded domain to one score (logit) per class, and a softmax turns the scores into probabilities: `dga` and `legit` in the binary setup, `legit` plus the DGA families in the family-first setup. In both, P(DGA) = 1 - P(legit).

In [ ]:
SETUPS = {"binary": "label", "family": "family"}   # setup -> target column


def make_predictor(task):
    """A new, untrained Model 2c with one output per class of `task`."""
    return CharCNN3({task: len(classes[task])}, width=WIDTH, hidden=HIDDEN, dropout=DROPOUT,
                    pool_dropout=POOL_DROPOUT)


def predict_p_dga(model, x, task):
    """P(DGA) = 1 - P(legit), from the softmax over the classes of `task`."""
    return p_dga(predict_proba(model, x), task, classes)


model = make_predictor("family")
print(model)
cnn2_size = sum(p.numel() for p in CharCNN2({"family": len(classes["family"])}, **CNN2).parameters())
print(f"{sum(p.numel() for p in model.parameters()):,} parameters (Model 2b: {cnn2_size:,})")

Before training, the weights are random, so the predictor cannot yet tell DGA from legitimate. As in Model 2b, the padding does not change its output: the same domains padded to 93 characters (as everywhere in this notebook) or to 40 get exactly the same P(DGA).

In [ ]:
examples = val.groupby("label").head(3)
model = make_predictor("label")
pd.DataFrame({"domain": examples["domain"].to_numpy(),
              "label": examples["label"].to_numpy(),
              "P(DGA), untrained": predict_p_dga(model, encode_split(examples["domain"]),
                                                 "label").round(3),
              "padded to 40": predict_p_dga(model, encode_split(examples["domain"], max_len=40),
                                            "label").round(3)})

## 2. Define a loss

The loss is the same as in 03 and 06. The loss of a batch of $B$ domains is the paper's loss (Table 2) with the sample weights added:

$$L = \frac{1}{B}\sum_{i=1}^{B} w_i \,\big(-\log P(y_i \mid x_i)\big) \;+\; \lambda_1 \sum |W| \;+\; \lambda_2 \sum W^2$$

- The first term is the cross-entropy: minus the log of the probability the predictor gives to the true class $y_i$ of domain $x_i$, weighted by its sample weight $w_i$. With `WEIGHTED = False`, every $w_i$ is 1.
- The other two terms are the L1 ($\lambda_1$ = 1e-5) and L2 ($\lambda_2$ = 1e-4) penalties on the convolution and dense weights $W$ (not the embedding, the biases or the batch normalization), added as Keras does (`penalty` in `src/models/train.py`).

In [ ]:
def loss_fn(model, logits, y, w, task):
    """Loss of one batch: weighted cross-entropy + L1/L2 penalty on the weights."""
    cross_entropy = F.cross_entropy(logits[task], y, reduction="none")   # -log P(y_i | x_i)
    return (cross_entropy * w).mean() + penalty(model, L1, L2)

The loss of the untrained binary predictor on the first batch of the training set:

In [ ]:
xb = torch.as_tensor(x_train[:BATCH_SIZE])
yb = torch.as_tensor(class_codes(train["label"].iloc[:BATCH_SIZE], classes["label"]))
wb = torch.as_tensor(np.array(train["weight"].iloc[:BATCH_SIZE]) if WEIGHTED
                     else np.ones(BATCH_SIZE), dtype=torch.float32)
with torch.no_grad():
    loss = loss_fn(model, model(xb), yb, wb, "label")
    reg = penalty(model, L1, L2)
print(f"loss {loss:.4f} = weighted cross-entropy {loss - reg:.4f} + penalty {reg:.4f}")

## 3. Optimize the loss

As in 06: Adam (learning rate `LR`) minimizes the loss on shuffled mini-batches of `BATCH_SIZE` domains, for `EPOCHS` epochs. After every epoch, the predictor is scored on the validation set, and the weights of the epoch with the highest macro-F1 are kept. `fit` turns both dropouts off and uses the batch-normalization statistics of the whole training (`model.eval()`) to score the validation set.

Three losses are recorded per epoch: $J_{train}$ (the mean loss over the epoch's training batches, with dropout on), $J_{val}$ (the loss on the whole validation set at the end of the epoch), and the **binary validation loss**, the weighted cross-entropy of the DGA/legitimate decision alone, the loss to compare the two setups and the three models with (see 06).

`optimize` puts the three pieces together: the predictor (step 1), the loss (step 2) and the Adam optimizer. The training loop that runs them is `fit` in `src/models/train.py`.

In [ ]:
tracking.setup(MLFLOW_EXPERIMENT)   # the trainings below are logged to this MLflow experiment

w_train = train["weight"].to_numpy() if WEIGHTED else None
w_val = sample_weights(val) if WEIGHTED else None


def optimize(setup):
    """Train a new predictor for `setup`: Adam on loss_fn, run by fit (src/models/train.py)."""
    task = SETUPS[setup]
    set_seed(SEED)
    model = make_predictor(task)                              # step 1: the predictor
    loss = partial(loss_fn, task=task)                        # step 2: the loss
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)   # step 3: the optimizer
    history, p = fit(
        model, loss, optimizer,
        train=Split(x_train, class_codes(train[task], classes[task]), w_train),
        val=Split(x_val, class_codes(val[task], classes[task]), w_val, y_val),
        decide=task, classes=classes, folder=RUN_DIRS[setup],
        epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED, device=device,
        params={"model": "charcnn3", "setup": setup, "task": task, "sample": SAMPLE,
                "lr": LR, "l1": L1, "l2": L2, "weighted": WEIGHTED, "width": WIDTH,
                "hidden": HIDDEN, "dropout": DROPOUT, "pool_dropout": POOL_DROPOUT})
    return model, history, p

One model per setup, each from the same seed. While a setup trains, its loss plot is redrawn after every epoch. After every epoch, `fit` saves the training in the setup's folder in `checkpoints/` (`RUN_DIRS`: `checkpoints_binary_cnn3_1` and `checkpoints_multifamily_cnn3_1` for `RUN = 1`): `epoch_01.pt`, ... (the weights after each epoch), `history.csv` (the losses and validation metrics of every epoch so far), `last.pt` (everything needed to continue) and, when the setup has finished, `best.pt` (the weights of the kept epoch). Each setup is also an MLflow run of the experiment `charcnn`, named after its folder.

If training stops (interrupted, crash, restart), run the notebook again from the top: each setup continues from its last saved epoch, and a setup that has finished is only reloaded. A run cannot be continued with different settings: change `RUN` to train from scratch.

In [ ]:
models, histories, p_val = {}, {}, {}
for setup, task in SETUPS.items():
    print(f"--- {setup} ({len(classes[task])} classes), saved in {RUN_DIRS[setup]}")
    models[setup], histories[setup], p_val[setup] = optimize(setup)

In [ ]:
# The histories are read from the saved history.csv files, so this cell also plots a
# training that was interrupted (the setups and epochs saved so far).
histories = {setup: load_history(folder) for setup, folder in RUN_DIRS.items()}
histories = {setup: history for setup, history in histories.items() if history is not None}

# The loss of each setup, one panel per setup: the family loss also counts the DGA
# families taken for one another, so the two losses are not on the same scale.
fig, axes = plt.subplots(1, len(histories), figsize=(6 * len(histories), 3.6), squeeze=False)
for ax, (setup, history) in zip(axes[0], histories.items()):
    plot_losses(ax, history, f"Loss, {setup}")
fig.tight_layout()
plt.show()

# The two setups compared on the DGA/legitimate decision: the binary validation loss
# (on the same scale in both setups) and the validation macro-F1.
panels = {"val_binary_loss": ("loss", "Validation loss on DGA vs. legitimate"),
          "macro_f1": ("macro-F1", "Validation macro-F1")}
fig, axes = plt.subplots(1, len(panels), figsize=(12, 3.6))
for ax, (column, (ylabel, title)) in zip(axes, panels.items()):
    for setup, history in histories.items():
        ax.plot(history.index, history[column], marker="o", label=setup)
    ax.set_xlabel("epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
fig.tight_layout()
plt.show()

## 4. Test the model

Each trained predictor gives P(DGA) for every domain of the evaluation set (`EVAL_SET`: the validation set for now, the test set for the final results). A domain is called DGA when P(DGA) >= 0.5. The metrics are those of the baselines, on the DGA/legitimate decision.

In [ ]:
# On the validation set, P(DGA) of the kept epoch was already computed in step 3 (same
# weights, same numbers), so the predictor only needs to run again on the test set.
p_dga_eval = {setup: p_val[setup] if EVAL_SET == "val" else predict_p_dga(models[setup], x_eval, task)
              for setup, task in SETUPS.items()}
results = {("Model 2c", setup): binary_metrics(y_eval, p) for setup, p in p_dga_eval.items()}
table = pd.DataFrame(results).T.round(4)
table.index.names = ["model", "setup"]
table

### Recall per DGA family

Recall of each DGA family in the two setups, families ordered from largest to smallest.

In [ ]:
family_sizes = train_full.loc[train_full["label"] == "dga", "family"].value_counts()
recall = pd.DataFrame({
    setup: recall_per_family(evaluate["family"], y_eval, (p >= 0.5).astype(int))
    for setup, p in p_dga_eval.items()}).reindex(family_sizes.index)
recall.columns = ["binary", "family-first"]

fig, ax = plt.subplots(figsize=(12, 3.6))
x = np.arange(len(recall))
for setup, marker in [("binary", "o"), ("family-first", "D")]:
    ax.scatter(x, recall[setup], s=24, marker=marker, label=setup, zorder=3)
ax.set_xlim(-1, len(recall))
ax.set_ylim(-0.03, 1.05)
ax.set_xlabel(f"DGA family, largest to smallest ({len(recall)} families)")
ax.set_ylabel("recall")
ax.set_title("Recall per DGA family, binary vs. family-first")
ax.legend(loc="lower left")
fig.tight_layout()
plt.show()

### Comparison with Model 2 (03) and Model 2b (06)

The trained Model 2 (`PAPER_DIRS`) and Model 2b (`CNN2_DIRS`) are scored on the same evaluation set, each with its own input (`encode`: no mark on the ending). The three models have the same data, setups, loss and training, so the differences come from the architecture. A model whose folder has no `best.pt` is left out.

In [ ]:
MODELS = {   # name -> (a new model for a task, its checkpoint folders, its input encoding)
    "Model 2": (lambda task: CharCNN({task: len(classes[task])}), PAPER_DIRS, encode),
    "Model 2b": (lambda task: CharCNN2({task: len(classes[task])}, **CNN2), CNN2_DIRS, encode),
    "Model 2c": (make_predictor, RUN_DIRS, encode_split)}


def load(name, setup):
    """The trained model `name` of `setup` (its kept epoch), or None if it is not trained."""
    make, dirs, _ = MODELS[name]
    if not (dirs[setup] / "best.pt").exists():
        return None
    model = make(SETUPS[setup]).to(device)
    model.load_state_dict(torch.load(dirs[setup] / "best.pt", map_location=device,
                                     weights_only=True))
    return model


p_all = {("Model 2c", setup): p for setup, p in p_dga_eval.items()}   # (model, setup) -> P(DGA)
for name in ("Model 2", "Model 2b"):
    x_input = MODELS[name][2](evaluate["domain"])
    for setup, task in SETUPS.items():
        model = load(name, setup)
        if model is None:
            print(f"{name}, {setup}: left out, no best.pt in {MODELS[name][1][setup]}")
            continue
        p_all[name, setup] = predict_p_dga(model, x_input, task)

comparison = pd.DataFrame({key: binary_metrics(y_eval, p) for key, p in p_all.items()}).T
comparison.index.names = ["model", "setup"]
comparison.sort_index().round(4)

### Did the changes work? Missed DGA domains and false alarms per name length

The changes aim at short names. If they work, Model 2c misses fewer short DGA names (FN) **and** calls fewer short legitimate names DGA (FP) than Model 2b, and on long names it does about as well as Model 2b. The tables count, per length of the name (the characters before the ending, as `encode_split` splits the domain), the DGA and legitimate domains, then for each model the DGA domains it misses and the legitimate domains it calls DGA (P(DGA) >= 0.5). One table per setup.

In [ ]:
length_band = pd.Series(pd.cut(name_length(x_eval), [0, 4, 6, 8, 12, 100],
                               labels=["1-4", "5-6", "7-8", "9-12", "13+"]), name="name length")
for setup in SETUPS:
    predictions = {name: (p >= 0.5).astype(int) for (name, s), p in sorted(p_all.items())
                   if s == setup}
    print(f"{setup} setup")
    display(errors_by_group(length_band, y_eval, predictions))

### Recall of the families Model 2b misses most

Recall of the 10 DGA families that Model 2b misses most in the family-first setup (those of Model 2c if Model 2b is left out), in every model and setup. The changes aim at the short-name families (qsnatch, virut, conficker, nymaim, proslikefan, lechiket); the word families (matsnu, nymaim2, gozi, bigviktor) are not expected to change.

In [ ]:
recalls = pd.DataFrame({key: recall_per_family(evaluate["family"], y_eval, (p >= 0.5).astype(int))
                        for key, p in sorted(p_all.items())})
recalls.columns.names = ["model", "setup"]
reference = ("Model 2b", "family") if ("Model 2b", "family") in recalls else ("Model 2c", "family")
recalls.loc[recalls[reference].sort_values().head(10).index].round(3)

Validation curves of every model, from their saved histories: the loss on the DGA/legitimate decision and the macro-F1 per epoch (dotted: Model 2, dashed: Model 2b, solid: Model 2c).

In [ ]:
styles = {"Model 2": ":", "Model 2b": "--", "Model 2c": "-"}
colors = {"binary": "C0", "family": "C1"}
fig, axes = plt.subplots(1, len(panels), figsize=(12, 3.6))
for ax, (column, (ylabel, title)) in zip(axes, panels.items()):
    for name, (_, dirs, _) in MODELS.items():
        for setup in SETUPS:
            history = load_history(dirs[setup])
            if history is not None:
                ax.plot(history.index, history[column], linestyle=styles[name],
                        color=colors[setup], label=f"{name}, {setup}")
    ax.set_xlabel("epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

### Latent space: Model 2b vs Model 2c

The latent space of a CNN is the vector its output layer reads: the output of its dense layer, 128 numbers per domain in both models (`src/models/latent.py`). The output layer is linear, so a domain whose latent vector sits among those of another class gets about the same prediction as them. The figures are those of 06, for Model 2b and Model 2c; see 06 for how to read them. Every model sees the same validation domains (up to 100 per DGA family, and as many legitimate domains), each with its own input, and what a domain is mixed with is measured by its 15 nearest neighbours in the full latent space.

What to look for: in the **DGA vs legitimate** maps, Model 2b's conflicts (ink points) are along the edge of the legitimate region, around qsnatch, virut, matsnu and nymaim2. If the changes work, there are fewer of them around the short-name families (qsnatch, virut), and fewer hollow circles (legitimate domains among DGA ones: likely false alarms); in the rankings, the aqua marker of Model 2c is to the left of the blue marker of Model 2b for those families (lower is better).

In [ ]:
COMPARED = ("Model 2b", "Model 2c")
sample = latent.sample_domains(evaluate, per_family=100, seed=SEED)   # the same for every model

spaces = {}   # setup -> {model name: its latent space}
for setup in SETUPS:
    loaded = {name: load(name, setup) for name in COMPARED}
    missing = [f"{name} ({MODELS[name][1][setup].name})" for name, m in loaded.items() if m is None]
    if missing:
        print(f"{setup}: skipped, not trained yet: {', '.join(missing)}")
        continue
    spaces[setup] = {name: latent.LatentSpace(latent.embed(m, MODELS[name][2](sample["domain"])),
                                              sample, seed=SEED)
                     for name, m in loaded.items()}
print(f"{len(sample):,} domains ({sample['family'].nunique() - 1} DGA families); "
      f"latent spaces of the {' and '.join(spaces) or 'no'} setup(s)")

In [ ]:
for setup, models_ in spaces.items():
    families = setup == "family"   # the family view only where the models learned the families
    plot_latent(models_, f"Latent space, {setup} setup: Model 2b vs Model 2c", families=families)
    plt.show()
    plot_conflicts(models_, families=families)
    plt.show()
    display(latent.verdict(models_, families=families))
    name, todo = latent.actions(models_, families=families)
    print(f"What to do ({setup} setup), from the latent space of {name}, "
          f"the better on DGA vs legitimate:")
    display(todo.style.hide(axis="index")
            .set_properties(**{"text-align": "left", "vertical-align": "top"}))

### How to read the result

The changes worked if, compared with Model 2b, in the family-first setup:

1. **Short names**: in the error tables, the missed DGA domains (FN) **and** the false alarms (FP) of the bands 1-4 and 5-6 both go down. One going down while the other goes up is a different threshold, not a better model.
2. **Long names**: the errors of the bands 9-12 and 13+ stay about the same (the changes are not aimed at them).
3. **Families**: the recall of qsnatch, virut and conficker goes up; matsnu, nymaim2, gozi and bigviktor (names made of words) stay about where they were.
4. **Overall**: macro-F1, precision and recall are at least those of Model 2b.

A change of a few domains in a small band or family can be chance: compare it with the size of the band (the `domains` columns). The model is trained once per setup, from one seed; a difference that matters should be checked with another seed (`SEED`, with a new `RUN`).